https://github.com/kirakom270199-dotcom/project_1

### Бизнес-контекст и задача проекта

Представьте, что вы работаете в соцсетевом приложении, где пользователи постят короткие тексты. В продукте стоит задача — добавить возможность автодополнения текстов. Разработчики просят вас создать модель, которую можно запускать на мобильных устройствах. Для смартфонов есть значительные требования по оперативной памяти и скорости работы, так что важна легковесность модели. 

### Формальная задача
Поэтапное описание задачи:
1. Взять датасет от разработчиков, очистить его, подготовить для обучения модели.
2. Реализовать и обучить модель на основе рекуррентных нейронных сетей.
3. Замерить качество разработанной и обученной модели.
4. Взять более «тяжёлую» предобученную модель из Transformers и замерить её качество.
5. Проанализировать результаты и дать рекомендации разработчикам: стоит ли использовать лёгкую модель или лучше постараться поработать с ограничениями по памяти и использовать большую предобученную.

#  Этап 0. Импорты

In [ ]:
pip install datasets

In [ ]:
pip install evaluate

In [ ]:
pip install rouge-score absl-py

In [ ]:
import re
import requests
import pandas as pd
import os
import random
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, GPT2LMHeadModel
from tqdm import tqdm
import os
import evaluate
from torch.nn.utils.rnn import pad_sequence

#  Этап 1. Сбор данных. Скачать датасет с короткими постами sentiment140.

In [ ]:
url = "https://code.s3.yandex.net/deep-learning/tweets.txt"

try:
    # Загружаем данные по URL
    response = requests.get(url)
    response.encoding = 'utf-8'
    response.raise_for_status()

    # Разделяем текст на строки
    tweets = response.text.splitlines()

    # Создаем DataFrame с сырыми данными
    raw_dataset = pd.DataFrame({'raw_text': tweets})

    print(f"Загружено твитов: {len(raw_dataset)}")

except requests.exceptions.RequestException as e:
    print(f"Ошибка при загрузке данных: {e}")
    raw_dataset = pd.DataFrame()

In [ ]:
raw_dataset.head(5)

# Этап 2. Подготовка данных

## 2.1. Нормализация

Привести к нижнему регистру;

удалить ссылки, упоминания, эмодзи (по необходимости);

заменить нестандартные символы;

In [ ]:
def clean_string(text):
    """Функция для очищения и предобрабатывания текста"""
    # приведение к нижнему регистру
    text = text.lower()
    # удаление всего, кроме латинских букв, цифр и пробелов
    text = re.sub(r'[^a-z0-9\s]', '', text)
    # удаление дублирующихся пробелов, удаление пробелов по краям
    text = re.sub(r'\s+', ' ', text).strip()

    return text

raw_dataset['cleaned_text'] = raw_dataset['raw_text'].apply(clean_string)
# Заполняем возможные NaN значения пустыми строками
raw_dataset['cleaned_text'] = raw_dataset['cleaned_text'].fillna('')
# Удаляем пустые строки после очистки
df_processed = raw_dataset[raw_dataset['cleaned_text'].str.len() > 0].copy()

In [ ]:
print(f"Осталось твитов после обработки: {len(df_processed)}")

## 2.2. Разобейм датасет на трейн, валидацию и тест.

In [ ]:
# Разделяем на train (80%) и temp (20%)
train_df, temp_df = train_test_split(df_processed, test_size=0.2, random_state=42)

# Разделяем temp на validation и test
# половина от temp = 10% от общего
val_df, test_df = train_test_split(temp_df, train_size=0.5, random_state=42)

print(len(train_df), len(val_df), len(test_df))

## 2.3. Токенизируем данных

In [ ]:
# Токенизатор
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")

def tokenize_texts(text_list):
    return tokenizer(text_list, truncation=True, padding=False)["input_ids"]

# Токенизация
train_tokens = tokenize_texts(train_df['cleaned_text'].tolist())
val_tokens = tokenize_texts(val_df['cleaned_text'].tolist())
test_tokens = tokenize_texts(test_df['cleaned_text'].tolist())

print(len(train_tokens), len(val_tokens), len(test_tokens))

## 2.4. Создание torch.Dataset и torch.DataLoader для обучения модели

In [ ]:
# 1. Dataset класс для обучения (X → Y)
class LanguageModelDataset(Dataset):
    def __init__(self, tokenized_texts, max_len=128):
        self.texts = []
        self.labels = []

        for tokens in tokenized_texts:
            if len(tokens) < 2:
                continue
            input_tokens = tokens[:-1][:max_len]
            target_tokens = tokens[1:][:max_len]
            min_len = min(len(input_tokens), len(target_tokens))
            if min_len > 0:
                self.texts.append(input_tokens[:min_len])
                self.labels.append(target_tokens[:min_len])

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return {
            'input_ids': torch.tensor(self.texts[idx], dtype=torch.long),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }


# Создание Datasets
train_dataset = LanguageModelDataset(train_tokens)
val_dataset = LanguageModelDataset(val_tokens)
test_dataset = LanguageModelDataset(test_tokens)

print(f"train_dataset: {len(train_dataset)}, val_dataset: {len(val_dataset)}, test_dataset: {len(test_dataset)}")

In [ ]:
# Collate функция
def collate_fn(batch):
    pad_token = tokenizer.pad_token_id or tokenizer.eos_token_id

    input_ids = [item['input_ids'] for item in batch]
    labels = [item['labels'] for item in batch]

    padded_inputs = pad_sequence(input_ids, batch_first=True, padding_value=pad_token)
    padded_labels = pad_sequence(labels, batch_first=True, padding_value=-100)
    attention_mask = (padded_inputs != pad_token).long()

    return {
        'input_ids': padded_inputs,
        'attention_mask': attention_mask,
        'labels': padded_labels
    }

# Создание DataLoader
batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

print(f"Количество батчей в train_loader: {len(train_loader)}, Количество батчей в val_loader: {len(val_loader)}, Количество батчей в test_loader: {len(test_loader)}")

# Освобождаем память:
del raw_dataset, df_processed, tweets
import gc
gc.collect()  # Принудительно запускаем сборщик мусора

print("Память после очистки исходных данных освобождена")

# Освобождаем DataFrame'ы:
del train_df, val_df, test_df
gc.collect()

print("Память после токенизации освобождена")

# Этап 3. Реализация рекуррентной сети

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden_size=128, pad_token_id=0):
        super().__init__()

        self.vocab_size = vocab_size
        self.pad_token_id = pad_token_id

        # Слои модели
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_token_id)
        self.lstm = nn.LSTM(emb_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, input_ids):
        # Эмбеддинг
        x = self.embedding(input_ids)  # [batch_size, seq_len, emb_dim]

        # LSTM
        rnn_out, _ = self.lstm(x)  # [batch_size, seq_len, hidden_size]

        # Выходные логиты
        logits = self.fc(rnn_out)  # [batch_size, seq_len, vocab_size]

        return logits

    @torch.no_grad()
    def generate_next_tokens(self, input_ids, max_new_tokens=50):
        """Простая генерация - всегда выбираем самый вероятный токен"""
        self.eval()
        generated = input_ids.clone()

        for _ in range(max_new_tokens):
            logits = self.forward(generated)  # [batch_size, seq_len, vocab_size]

            # Берем логиты для последнего токена и выбираем самый вероятный
            next_token_logits = logits[:, -1, :]  # [batch_size, vocab_size]
            next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)  # [batch_size, 1]

            # Добавляем к сгенерированной последовательности
            generated = torch.cat([generated, next_token], dim=1)

            # Останавливаемся если достигли pad token
            if (next_token == self.pad_token_id).all():
                break

        return generated

In [ ]:
# Определяем параметры
vocab_size = len(tokenizer)  # размер словаря distilgpt2
pad_token_id = tokenizer.pad_token_id or tokenizer.eos_token_id

# Создаем простую модель
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LSTMModel(
    vocab_size=vocab_size,
    emb_dim=64,
    hidden_size=64,
    pad_token_id=pad_token_id
).to(device)

print(f"Простая LSTM модель создана")
print(f"Устройство: {device}")

# Этап 3. Тренировка модели

In [ ]:
# Инициализация ClearML task
try:
    from clearml import Task
    task = Task.init(project_name="LSTM Language Model", task_name="LSTM Training with ROUGE")
    USE_CLEARML = True
except ImportError:
    print("ClearML not available, continuing without logging")
    USE_CLEARML = False

# Конфигурация
config = {
    "learning_rate": 1e-3,
    "batch_size": 32,
    "epochs": 1,
    "embedding_dim": 64,
    "hidden_dim": 64,
    "max_grad_norm": 1.0,
    "patience": 3
}

if USE_CLEARML:
    task.connect(config)

# Используем вашу существующую модель
vocab_size = len(tokenizer)
pad_token_id = tokenizer.pad_token_id or tokenizer.eos_token_id
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Используемое устройство: {device}")
print(f"Размер словаря: {vocab_size}")

# Создаем модель (используем вашу существующую)
model = LSTMModel(
    vocab_size=vocab_size,
    emb_dim=64,
    hidden_size=64,
    pad_token_id=pad_token_id
).to(device)

# Оптимизатор и функция потерь
optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"])
criterion = nn.CrossEntropyLoss(ignore_index=-100)

# Загрузка метрики ROUGE
rouge = evaluate.load("rouge")

# Вспомогательные функции
def tokens_to_text(tokens):
    """Конвертирует токены в текст"""
    if isinstance(tokens, torch.Tensor):
        toks = tokens.cpu().tolist()
    else:
        toks = list(tokens)
    toks = [t for t in toks if isinstance(t, int) and t >= 0 and t != -100 and t != pad_token_id]
    return tokenizer.decode(toks, skip_special_tokens=True, clean_up_tokenization_spaces=True)

@torch.no_grad()
def greedy_generate(model, prefix_ids, gen_len, device):
    """Жадная генерация текста"""
    model.eval()
    if gen_len <= 0:
        return []
    prefix_tensor = torch.tensor(prefix_ids, dtype=torch.long, device=device).unsqueeze(0)
    generated = prefix_tensor.clone()
    for _ in range(gen_len):
        logits = model(generated)
        next_id = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
        generated = torch.cat([generated, next_id], dim=1)
    return generated.squeeze(0).cpu().tolist()

# Функции тренировки и валидации
def train_one_epoch(model, loader, optimizer, criterion, device, epoch_num):
    """Тренировка одной эпохи с автоматической смешанной точностью"""
    model.train()
    total_loss = 0
    progress_bar = tqdm(loader, desc=f"Epoch {epoch_num} Training")

    # ✅ Универсальный способ (работает во всех версиях PyTorch)
    try:
        scaler = torch.amp.GradScaler('cuda')
    except Exception:
        scaler = torch.cuda.amp.GradScaler()

    for batch_idx, batch in enumerate(progress_bar):
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        # ✅ Автоматическая смешанная точность (также совместимая запись)
        try:
            autocast_context = torch.amp.autocast('cuda')
        except Exception:
            autocast_context = torch.cuda.amp.autocast()

        with autocast_context:
            logits = model(input_ids)
            V = logits.size(-1)
            loss = criterion(logits.view(-1, V), labels.view(-1))

        # ✅ backward и шаг оптимизатора через scaler
        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), config["max_grad_norm"])
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        torch.cuda.empty_cache()

        # Прогресс-бар
        if batch_idx % 10 == 0:
            avg_loss = total_loss / (batch_idx + 1)
            progress_bar.set_postfix({"loss": f"{avg_loss:.4f}"})

    return total_loss / len(loader)

@torch.no_grad()
def validate_loss(model, loader, criterion, device):
    """Валидация loss"""
    model.eval()
    total_loss = 0.0
    for batch in tqdm(loader, desc="Validation Loss"):
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        logits = model(input_ids)
        V = logits.size(-1)
        loss = criterion(logits.view(-1, V), labels.view(-1))
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def validate_generation(model, loader, device, max_examples=200, print_examples=3):
    """Валидация генерации с ROUGE метриками"""
    model.eval()
    preds, refs = [], []
    printed = 0

    for batch in tqdm(loader, desc="Validation Generation"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch.get('attention_mask', None)
        labels = batch.get('labels', None)

        B, S = input_ids.size()

        # Используем attention_mask для определения реальной длины
        if attention_mask is not None:
            real_lens = attention_mask.sum(dim=1).cpu().tolist()
        else:
            real_lens = [S] * B

        for i in range(B):
            if len(preds) >= max_examples:
                break

            real_len = int(real_lens[i])
            if real_len < 4:  # Слишком короткие последовательности пропускаем
                continue

            # Берем 3/4 текста как префикс, 1/4 как цель
            cut = max(1, (3 * real_len) // 4)
            prefix_ids = input_ids[i, :cut].cpu().tolist()

            # Используем labels если есть, иначе исходный input_ids
            if labels is not None:
                ref_tail_ids = labels[i, :real_len].cpu().tolist()
            else:
                ref_tail_ids = input_ids[i, cut:real_len].cpu().tolist()

            # Фильтруем -100 в labels
            ref_tail_ids = [t for t in ref_tail_ids if t != -100]

            gen_len = len(ref_tail_ids)
            if gen_len <= 0:
                continue

            # Генерируем продолжение
            gen_full = greedy_generate(model, prefix_ids, gen_len, device)
            gen_tail = gen_full[len(prefix_ids):] if len(gen_full) > len(prefix_ids) else []

            pred_text = tokens_to_text(gen_tail)
            ref_text = tokens_to_text(ref_tail_ids)

            if len(ref_text.strip()) == 0:
                continue

            preds.append(pred_text)
            refs.append(ref_text)

            # Выводим примеры
            if printed < print_examples:
                print("\n" + "="*50)
                print(f"ПРИМЕР ГЕНЕРАЦИИ {printed + 1}:")
                print("="*50)
                print(f"ПРЕФИКС: {tokens_to_text(prefix_ids)}")
                print(f"ОЖИДАЕМОЕ ПРОДОЛЖЕНИЕ: {ref_text}")
                print(f"СГЕНЕРИРОВАННОЕ ПРОДОЛЖЕНИЕ: {pred_text}")
                printed += 1

        if len(preds) >= max_examples:
            break

    # Вычисляем ROUGE метрики
    rouge_scores = rouge.compute(predictions=preds, references=refs) if len(preds) > 0 else {}
    return rouge_scores

# Основной цикл тренировки
N_EPOCHS = config["epochs"]
MODEL_SAVE_PATH = "trained_lstm_model.pth"
os.makedirs("models", exist_ok=True)

print("Начинаем обучение...")
print(f"Размер тренировочного датасета: {len(train_loader.dataset)}")
print(f"Размер валидационного датасета: {len(val_loader.dataset)}")
print(f"Количество параметров модели: {sum(p.numel() for p in model.parameters()):,}")

best_val_loss = float('inf')
patience_counter = 0

for epoch in range(1, N_EPOCHS + 1):
    print(f"\n{'='*60}")
    print(f"ЭПОХА {epoch}/{N_EPOCHS}")
    print(f"{'='*60}")

    # Тренировка
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device, epoch)
    print(f"Train Loss: {train_loss:.4f}")

    # Валидация loss
    val_loss = validate_loss(model, val_loader, criterion, device)
    print(f"Val Loss:   {val_loss:.4f}")

    # Валидация генерации с ROUGE
    rouge_scores = validate_generation(
        model, val_loader, device,
        max_examples=200, print_examples=3
    )

    if rouge_scores:
        print(f"\nROUGE Метрики:")
        for metric, score in rouge_scores.items():
            print(f"  {metric}: {score:.4f}")
    else:
        print("ROUGE: не удалось вычислить метрики")

    # Логирование в ClearML
    if USE_CLEARML:
        try:
            task.get_logger().report_scalar(
                title="Loss", series="Train", iteration=epoch, value=float(train_loss)
            )
            task.get_logger().report_scalar(
                title="Loss", series="Val", iteration=epoch, value=float(val_loss)
            )
            for metric_name, value in (rouge_scores or {}).items():
                task.get_logger().report_scalar(
                    title=f"ROUGE", series=metric_name, iteration=epoch, value=float(value)
                )
        except Exception as e:
            print(f"ClearML logging error: {e}")

    # Early stopping и сохранение лучшей модели
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"✅ Модель улучшилась! Сохранена в {MODEL_SAVE_PATH}")
    else:
        patience_counter += 1
        print(f"🚫 Patience counter: {patience_counter}/{config['patience']}")

    if patience_counter >= config["patience"]:
        print("🛑 Early stopping triggered!")
        break

print(f"\n{'='*60}")
print("ОБУЧЕНИЕ ЗАВЕРШЕНО!")
print(f"{'='*60}")

# Загрузка лучшей модели и финальное тестирование
print("\nЗагружаем лучшую модель для финального тестирования...")
model.load_state_dict(torch.load(MODEL_SAVE_PATH))

# Финальное тестирование на test set
print("\nФИНАЛЬНОЕ ТЕСТИРОВАНИЕ НА TEST SET:")
test_loss = validate_loss(model, test_loader, criterion, device)
final_rouge = validate_generation(model, test_loader, device, max_examples=500, print_examples=5)

print(f"\nФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ:")
print(f"Test Loss: {test_loss:.4f}")
if final_rouge:
    print("Test ROUGE Метрики:")
    for metric, score in final_rouge.items():
        print(f"  {metric}: {score:.4f}")
else:
    print("Test ROUGE: не удалось вычислить метрики")

print(f"\nМодель успешно обучена и сохранена в: {MODEL_SAVE_PATH}")
print(f"Лучшая Val Loss: {best_val_loss:.4f}")

ClearML not available, continuing without logging
Используемое устройство: cuda
Размер словаря: 50257
Начинаем обучение...
Размер тренировочного датасета: 1280103
Размер валидационного датасета: 160000
Количество параметров модели: 6,516,433

============================================================
ЭПОХА 1/1
============================================================
Epoch 1 Training: 100%|██████████| 20002/20002 [13:16<00:00, 25.12it/s, loss=6.9880]
Train Loss: 6.9880
Validation Loss: 100%|██████████| 2500/2500 [00:45<00:00, 55.07it/s]
Val Loss:   6.7117
Validation Generation:   0%|          | 0/2500 [00:00<?, ?it/s]

==================================================
ПРИМЕР ГЕНЕРАЦИИ 1:
==================================================
ПРЕФИКС: how the hell did this night start with tania crying to me crying wtf
ОЖИДАЕМОЕ ПРОДОЛЖЕНИЕ:  the hell did this night start with tania crying to me crying wtf ive fuckin had enough of this
СГЕНЕРИРОВАННОЕ ПРОДОЛЖЕНИЕ:  and i have a good night i have to be a good night i have to be a good night i have

==================================================
ПРИМЕР ГЕНЕРАЦИИ 2:
==================================================
ПРЕФИКС: robkardashian
ОЖИДАЕМОЕ ПРОДОЛЖЕНИЕ: kardashian gooooood
СГЕНЕРИРОВАННОЕ ПРОДОЛЖЕНИЕ:  i have a good night i have

==================================================
ПРИМЕР ГЕНЕРАЦИИ 3:
==================================================
ПРЕФИКС: my stupid brother woke me up
ОЖИДАЕМОЕ ПРОДОЛЖЕНИЕ:  stupid brother woke me up at like 730
СГЕНЕРИРОВАННОЕ ПРОДОЛЖЕНИЕ:  to be a good night i have to be
Validation Generation:   0%|          | 3/2500 [00:01<17:27,  2.38it/s]

ROUGE Метрики:
  rouge1: 0.0828
  rouge2: 0.0092
  rougeL: 0.0809
  rougeLsum: 0.0808
✅ Модель улучшилась! Сохранена в trained_lstm_model.pth

============================================================
ОБУЧЕНИЕ ЗАВЕРШЕНО!
============================================================

Загружаем лучшую модель для финального тестирования...

ФИНАЛЬНОЕ ТЕСТИРОВАНИЕ НА TEST SET:
Validation Loss: 100%|██████████| 2501/2501 [00:45<00:00, 55.20it/s]
Validation Generation:   0%|          | 0/2501 [00:00<?, ?it/s]

==================================================
ПРИМЕР ГЕНЕРАЦИИ 1:
==================================================
ПРЕФИКС: ooh i want to learn
ОЖИДАЕМОЕ ПРОДОЛЖЕНИЕ: oh i want to learn new swear words
СГЕНЕРИРОВАННОЕ ПРОДОЛЖЕНИЕ:  the best i have to be a good

==================================================
ПРИМЕР ГЕНЕРАЦИИ 2:
==================================================
ПРЕФИКС: hoping jt wins survivor
ОЖИДАЕМОЕ ПРОДОЛЖЕНИЕ: ing jt wins survivor lt3333
СГЕНЕРИРОВАННОЕ ПРОДОЛЖЕНИЕ:  i have a good night i have to

==================================================
ПРИМЕР ГЕНЕРАЦИИ 3:
==================================================
ПРЕФИКС: yes they were indeed motherlickers the scummy kind feel dirty for even going to the interview hhh
ОЖИДАЕМОЕ ПРОДОЛЖЕНИЕ:  they were indeed motherlickers the scummy kind feel dirty for even going to the interview hhhmmmmmm frickin fake yuppies
СГЕНЕРИРОВАННОЕ ПРОДОЛЖЕНИЕ:  i have a good night i have to be a good night i have to be a good night i have to be a good night i have to

==================================================
ПРИМЕР ГЕНЕРАЦИИ 4:
==================================================
...
==================================================
ПРЕФИКС: i fancy this boy but he f
ОЖИДАЕМОЕ ПРОДОЛЖЕНИЕ:  fancy this boy but he fancies my twin sister
СГЕНЕРИРОВАННОЕ ПРОДОЛЖЕНИЕ:  i have a good night i have to be a
Output is truncated. View as a scrollable element or open in a text editor. Adjust cell output settings...
Validation Generation:   0%|          | 8/2501 [00:02<13:05,  3.17it/s]

ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ:
Test Loss: 6.7125
Test ROUGE Метрики:
  rouge1: 0.0835
  rouge2: 0.0049
  rougeL: 0.0810
  rougeLsum: 0.0808

Модель успешно обучена и сохранена в: trained_lstm_model.pth
Лучшая Val Loss: 6.7117

In [ ]:
transformers_model_name = "distilgpt2"

try:
    transformers_tokenizer = AutoTokenizer.from_pretrained(transformers_model_name)
    transformers_model = AutoModelForCausalLM.from_pretrained(transformers_model_name)

    # Добавляем pad token если его нет
    if transformers_tokenizer.pad_token is None:
        transformers_tokenizer.pad_token = transformers_tokenizer.eos_token

    transformers_model.eval()
    print("✅ DistilGPT2 успешно загружен!")

except Exception as e:
    print(f"❌ Ошибка загрузки модели: {e}")

# Функция для генерации текста с помощью трансформера
@torch.no_grad()
def transformers_generate(text_prefix, max_new_tokens=50, temperature=0.7):
    """Генерация продолжения текста с помощью предобученного трансформера"""
    inputs = transformers_tokenizer.encode(text_prefix, return_tensors="pt")

    # Генерация с разнообразием
    outputs = transformers_model.generate(
        inputs,
        max_length=inputs.shape[1] + max_new_tokens,
        num_return_sequences=1,
        temperature=temperature,
        do_sample=True,
        pad_token_id=transformers_tokenizer.eos_token_id,
        no_repeat_ngram_size=2,
        early_stopping=True
    )

    generated_text = transformers_tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Извлекаем только сгенерированную часть (без префикса)
    generated_part = generated_text[len(text_prefix):].strip()

    return generated_part

# Функция для валидации трансформера
@torch.no_grad()
def validate_transformers(loader, tokenizer, max_examples=200, print_examples=5):
    """Валидация предобученного трансформера"""
    print("\n🧪 Запускаем валидацию предобученного трансформера...")

    rouge = evaluate.load("rouge")
    preds, refs = [], []
    printed = 0

    for batch_idx, batch in enumerate(tqdm(loader, desc="Transformers Validation")):
        if len(preds) >= max_examples:
            break

        input_ids = batch['input_ids']
        attention_mask = batch.get('attention_mask', None)
        labels = batch.get('labels', None)

        B, S = input_ids.size()

        # Используем attention_mask для определения реальной длины
        if attention_mask is not None:
            real_lens = attention_mask.sum(dim=1).cpu().tolist()
        else:
            real_lens = [S] * B

        for i in range(B):
            if len(preds) >= max_examples:
                break

            real_len = int(real_lens[i])
            if real_len < 10:  # Пропускаем слишком короткие последовательности
                continue

            # Берем 3/4 текста как префикс, 1/4 как цель
            cut = max(1, (3 * real_len) // 4)
            prefix_ids = input_ids[i, :cut]

            # Используем labels если есть, иначе исходный input_ids
            if labels is not None:
                ref_tail_ids = labels[i, :real_len]
            else:
                ref_tail_ids = input_ids[i, cut:real_len]

            # Фильтруем -100 в labels
            ref_tail_ids = ref_tail_ids[ref_tail_ids != -100]

            gen_len = len(ref_tail_ids)
            if gen_len <= 0:
                continue

            # Конвертируем в текст
            prefix_text = tokenizer.decode(prefix_ids.cpu().tolist(), skip_special_tokens=True)
            ref_text = tokenizer.decode(ref_tail_ids.cpu().tolist(), skip_special_tokens=True)

            if len(ref_text.strip()) == 0:
                continue

            # Генерируем продолжение с помощью трансформера
            try:
                pred_text = transformers_generate(prefix_text, max_new_tokens=gen_len)

                preds.append(pred_text)
                refs.append(ref_text)

                # Выводим примеры
                if printed < print_examples:
                    print("\n" + "="*60)
                    print(f"ПРИМЕР ГЕНЕРАЦИИ TRANSFORMER {printed + 1}:")
                    print("="*60)
                    print(f"ПРЕФИКС: {prefix_text}")
                    print(f"ОЖИДАЕМОЕ ПРОДОЛЖЕНИЕ: {ref_text}")
                    print(f"СГЕНЕРИРОВАННОЕ ПРОДОЛЖЕНИЕ: {pred_text}")
                    printed += 1

            except Exception as e:
                print(f"Ошибка генерации: {e}")
                continue

    # Вычисляем ROUGE метрики
    if len(preds) > 0:
        rouge_scores = rouge.compute(predictions=preds, references=refs)
        print(f"\n✅ Проверено примеров: {len(preds)}")
        return rouge_scores, preds, refs
    else:
        print("❌ Не удалось сгенерировать примеры для валидации")
        return {}, [], []

# Функция для сравнения двух моделей
def compare_models(lstm_model, transformers_model, test_loader, lstm_tokenizer, transformers_tokenizer, device):
    """Сравнение LSTM и Transformer моделей"""
    print("\n" + "="*80)
    print("СРАВНЕНИЕ МОДЕЛЕЙ: LSTM vs TRANSFORMER")
    print("="*80)

    # Тестируем LSTM модель
    print("\n1. ТЕСТИРУЕМ LSTM МОДЕЛЬ...")
    lstm_rouge = validate_generation(  # ← ИЗМЕНИТЬ ЗДЕСЬ: получаем только один результат
        lstm_model, test_loader, device, max_examples=100, print_examples=3
    )

    # Тестируем Transformer модель
    print("\n2. ТЕСТИРУЕМ TRANSFORMER МОДЕЛЬ...")
    transformers_rouge, transformers_preds, transformers_refs = validate_transformers(
        test_loader, lstm_tokenizer, max_examples=100, print_examples=3
    )

    # Сравниваем метрики
    print("\n" + "="*80)
    print("РЕЗУЛЬТАТЫ СРАВНЕНИЯ:")
    print("="*80)

    if lstm_rouge and transformers_rouge:
        print("\n📊 ROUGE МЕТРИКИ:")
        print(f"{'Метрика':<10} {'LSTM':<8} {'Transformer':<12} {'Разница':<10}")
        print("-" * 45)

        for metric in ['rouge1', 'rouge2', 'rougeL']:
            lstm_score = lstm_rouge.get(metric, 0)
            transformer_score = transformers_rouge.get(metric, 0)
            difference = transformer_score - lstm_score
            print(f"{metric:<10} {lstm_score:.4f}    {transformer_score:.4f}       {difference:+.4f}")

    # Сравниваем размеры моделей
    lstm_params = sum(p.numel() for p in lstm_model.parameters())
    transformer_params = sum(p.numel() for p in transformers_model.parameters())

    print(f"\n📏 РАЗМЕРЫ МОДЕЛЕЙ:")
    print(f"LSTM: {lstm_params:,} параметров")
    print(f"Transformer: {transformer_params:,} параметров")
    print(f"Отношение: {transformer_params/lstm_params:.1f}x")

    return {
        'lstm': {'rouge': lstm_rouge, 'params': lstm_params},
        'transformer': {'rouge': transformers_rouge, 'params': transformer_params}
    }


# Запускаем этап 4
if __name__ == "__main__":
    # Проверяем, что LSTM модель обучена и загружена
    try:
        # Загружаем обученную LSTM модель
        MODEL_SAVE_PATH = "trained_lstm_model.pth"
        lstm_model_loaded = LSTMModel(
            vocab_size=len(tokenizer),
            emb_dim=64,
            hidden_size=64,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id
        ).to(device)

        lstm_model_loaded.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
        lstm_model_loaded.eval()
        print("✅ Обученная LSTM модель загружена!")

    except Exception as e:
        print(f"❌ Не удалось загрузить LSTM модель: {e}")
        print("Запустите этап 3 сначала для обучения модели.")
        exit()

    # 1. Сравниваем модели на тестовом наборе
    comparison_results = compare_models(
        lstm_model_loaded,
        transformers_model,
        test_loader,
        tokenizer,
        transformers_tokenizer,
        device
    )

    # 3. Вывод рекомендаций для разработчиков
    print("\n" + "="*80)
    print("РЕКОМЕНДАЦИИ ДЛЯ РАЗРАБОТЧИКОВ")
    print("="*80)

    lstm_rouge1 = comparison_results['lstm']['rouge'].get('rouge1', 0)
    transformer_rouge1 = comparison_results['transformer']['rouge'].get('rouge1', 0)

    lstm_params = comparison_results['lstm']['params']
    transformer_params = comparison_results['transformer']['params']

    print(f"\n📈 КАЧЕСТВО (ROUGE-1):")
    print(f"LSTM: {lstm_rouge1:.4f}")
    print(f"Transformer: {transformer_rouge1:.4f}")

    print(f"\n💾 РАЗМЕР:")
    print(f"LSTM: {lstm_params:,} параметров")
    print(f"Transformer: {transformer_params:,} параметров")

    quality_ratio = transformer_rouge1 / lstm_rouge1 if lstm_rouge1 > 0 else float('inf')
    size_ratio = transformer_params / lstm_params

    print(f"\n⚖️ СООТНОШЕНИЕ:")
    print(f"Качество: {quality_ratio:.2f}x")
    print(f"Размер: {size_ratio:.1f}x")

    # Рекомендации
    print(f"\n💡 РЕКОМЕНДАЦИИ:")

    if quality_ratio > 1.5 and size_ratio < 10:
        print("✅ Рекомендуется использовать Transformer модель:")
        print("   - Значительно лучшее качество генерации")
        print("   - Приемлемый размер для современных смартфонов")
        print("   - Лучшая обработка контекста и грамматики")
    elif quality_ratio > 1.2:
        print("⚖️ Рассмотрите оптимизацию Transformer модели:")
        print("   - Качество лучше, но размер большой")
        print("   - Можно использовать квантизацию или прунинг")
        print("   - Или поискать более легкие версии Transformer")
    else:
        print("✅ Рекомендуется использовать LSTM модель:")
        print("   - Достаточное качество для автодополнения")
        print("   - Значительно меньший размер")
        print("   - Быстрее работает на мобильных устройствах")
        print("   - Меньше потребляет памяти и батареи")

    print(f"\n🎯 ВЫВОД:")
    print("Для мобильного приложения с ограничениями по памяти LSTM может быть")
    print("оптимальным выбором, обеспечивая баланс между качеством и производительностью.")

print("\n" + "="*80)
print("ЭТАП 4 ЗАВЕРШЕН!")
print("="*80)

✅ DistilGPT2 успешно загружен!
✅ Обученная LSTM модель загружена!

================================================================================
СРАВНЕНИЕ МОДЕЛЕЙ: LSTM vs TRANSFORMER
================================================================================

1. ТЕСТИРУЕМ LSTM МОДЕЛЬ...
Validation Generation:   0%|          | 0/2501 [00:00<?, ?it/s]

==================================================
ПРИМЕР ГЕНЕРАЦИИ 1:
==================================================
ПРЕФИКС: ooh i want to learn
ОЖИДАЕМОЕ ПРОДОЛЖЕНИЕ: oh i want to learn new swear words
СГЕНЕРИРОВАННОЕ ПРОДОЛЖЕНИЕ:  the best i have to be a good

==================================================
ПРИМЕР ГЕНЕРАЦИИ 2:
==================================================
ПРЕФИКС: hoping jt wins survivor
ОЖИДАЕМОЕ ПРОДОЛЖЕНИЕ: ing jt wins survivor lt3333
СГЕНЕРИРОВАННОЕ ПРОДОЛЖЕНИЕ:  i have a good night i have to

==================================================
ПРИМЕР ГЕНЕРАЦИИ 3:
==================================================
ПРЕФИКС: yes they were indeed motherlickers the scummy kind feel dirty for even going to the interview hhh
ОЖИДАЕМОЕ ПРОДОЛЖЕНИЕ:  they were indeed motherlickers the scummy kind feel dirty for even going to the interview hhhmmmmmm frickin fake yuppies
СГЕНЕРИРОВАННОЕ ПРОДОЛЖЕНИЕ:  i have a good night i have to be a good night i have to be a good night i have to be a good night i have to
Validation Generation:   0%|          | 1/2501 [00:00<22:51,  1.82it/s]

2. ТЕСТИРУЕМ TRANSFORMER МОДЕЛЬ...

🧪 Запускаем валидацию предобученного трансформера...
Transformers Validation:   0%|          | 0/2501 [00:00<?, ?it/s]

============================================================
ПРИМЕР ГЕНЕРАЦИИ TRANSFORMER 1:
============================================================
ПРЕФИКС: yes they were indeed motherlickers the scummy kind feel dirty for even going to the interview hhh
ОЖИДАЕМОЕ ПРОДОЛЖЕНИЕ:  they were indeed motherlickers the scummy kind feel dirty for even going to the interview hhhmmmmmm frickin fake yuppies
СГЕНЕРИРОВАННОЕ ПРОДОЛЖЕНИЕ: .
I don't mean to say that I disagree with them, I just mean, they've done a lot of fucking great job getting me

============================================================
ПРИМЕР ГЕНЕРАЦИИ TRANSFORMER 2:
============================================================
ПРЕФИКС: larkvamp oh my i have no special
ОЖИДАЕМОЕ ПРОДОЛЖЕНИЕ: arkvamp oh my i have no special squid cures for stomach problems
СГЕНЕРИРОВАННОЕ ПРОДОЛЖЕНИЕ: power.

I'm a big fan of my game. I

============================================================
ПРИМЕР ГЕНЕРАЦИИ TRANSFORMER 3:
============================================================
ПРЕФИКС: i fancy this boy but he f
ОЖИДАЕМОЕ ПРОДОЛЖЕНИЕ:  fancy this boy but he fancies my twin sister
СГЕНЕРИРОВАННОЕ ПРОДОЛЖЕНИЕ: idgeted from the roof. I want to go
Transformers Validation:   0%|          | 3/2501 [00:33<7:48:14, 11.25s/it] 

✅ Проверено примеров: 100

================================================================================
РЕЗУЛЬТАТЫ СРАВНЕНИЯ:
================================================================================

📊 ROUGE МЕТРИКИ:
Метрика    LSTM     Transformer  Разница   
---------------------------------------------
rouge1     0.0876    0.1267       +0.0391
rouge2     0.0062    0.0052       -0.0010
rougeL     0.0848    0.0945       +0.0097

📏 РАЗМЕРЫ МОДЕЛЕЙ:
LSTM: 6,516,433 параметров
Transformer: 81,912,576 параметров
Отношение: 12.6x

================================================================================
РЕКОМЕНДАЦИИ ДЛЯ РАЗРАБОТЧИКОВ
================================================================================

📈 КАЧЕСТВО (ROUGE-1):
LSTM: 0.0876
...

================================================================================
ЭТАП 4 ЗАВЕРШЕН!
================================================================================